## 1. Installation

In [1]:
!pip install -q -U transformers accelerate "bitsandbytes>=0.46.1" peft openpyxl
print("✅ Installation terminée. Maintenant : Exécution → Redémarrer la session")

✅ Installation terminée. Maintenant : Exécution → Redémarrer la session


## 1b. Vérification

In [2]:
import torch, transformers, bitsandbytes, peft
print("transformers :", transformers.__version__, "| bitsandbytes :", bitsandbytes.__version__, "| peft :", peft.__version__)
assert torch.cuda.is_available(), "⚠️ Pas de GPU : Exécution → Modifier le type d'exécution → GPU"
GPU = torch.cuda.get_device_name(0)
NATIF_BF16 = torch.cuda.get_device_capability(0)[0] >= 8      # L4 / A100 : oui ; T4 : non
print("GPU :", GPU, f"({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} Go)", "| bfloat16 natif :", NATIF_BF16)
if not NATIF_BF16:
    print("ℹ️ GPU T4 : Gemma tournera en bfloat16 émulé (plus lent, mais résultats corrects).")

transformers : 5.24.1 | bitsandbytes : 0.50.2 | peft : 0.22.0
GPU : Tesla T4 (16 Go) | bfloat16 natif : False
ℹ️ GPU T4 : Gemma tournera en bfloat16 émulé (plus lent, mais résultats corrects).


## 2. Connexion à Hugging Face
Obligatoire ici : `aya-expanse-8b` est à accès restreint.
Collez un token *Read* (huggingface.co → Settings → Access Tokens) **du compte qui a accepté les conditions d'Aya**.

In [3]:
from huggingface_hub import login
try:
    login()
except Exception as e:
    print("Connexion ignorée :", e)

## 3. Google Drive : les fichiers des 3 anciens modèles

In [4]:
import os, math, re, time, gc, shutil
import pandas as pd
from google.colab import drive, files

drive.mount("/content/drive")
DOSSIER = "/content/drive/MyDrive/SmartCampus_LLM"
os.makedirs(DOSSIER, exist_ok=True)

FICHIER_RESULTATS = os.path.join(DOSSIER, "resultats.csv")              # réponses (SA-33 / SA-34)
FICHIER_PPL = os.path.join(DOSSIER, "perplexite_details.csv")           # perplexité phrase par phrase

if os.path.exists(FICHIER_RESULTATS):
    resultats = pd.read_csv(FICHIER_RESULTATS, dtype={"id": str}).fillna("").to_dict("records")
    print(f"♻️ Réponses déjà enregistrées : {len(resultats)}")
    print(pd.DataFrame(resultats).groupby("modele").size().to_string(), "\n")
else:
    resultats = []
    print("⚠️ resultats.csv introuvable : seuls les 2 nouveaux modèles seront dans les tableaux.\n")

if os.path.exists(FICHIER_PPL):
    details = pd.read_csv(FICHIER_PPL, dtype={"phrase_id": str}).to_dict("records")
    print(f"♻️ Phrases de perplexité déjà calculées : {len(details)}")
    print(pd.DataFrame(details).groupby("modele").size().to_string())
else:
    details = []
    print("⚠️ perplexite_details.csv introuvable : seuls les 2 nouveaux modèles seront dans les tableaux.")

def sauvegarder():
    pd.DataFrame(resultats).to_csv(FICHIER_RESULTATS, index=False)

def sauvegarder_ppl():
    pd.DataFrame(details).to_csv(FICHIER_PPL, index=False)

Mounted at /content/drive
♻️ Réponses déjà enregistrées : 108
modele
Artess99/derja-qwen3-8b-derja-v2    36
Qwen/Qwen2.5-7B-Instruct            36
Qwen/Qwen3-8B                       36 

♻️ Phrases de perplexité déjà calculées : 600
modele
Artess99/derja-qwen3-8b-derja-v2    200
Qwen/Qwen2.5-7B-Instruct            200
Qwen/Qwen3-8B                       200


## 4. Les données officielles des formations (identique au notebook de comparaison)

In [5]:
def lire(chemin):
    if chemin.lower().endswith((".xlsx", ".xls")):
        return pd.read_excel(chemin, dtype=str).fillna("")
    return pd.read_csv(chemin, dtype=str, encoding="utf-8").fillna("")

existant = [f for f in ("formations.xlsx", "formations.csv") if os.path.exists(os.path.join(DOSSIER, f))]
if existant:
    formations = lire(os.path.join(DOSSIER, existant[0]))
    print("Fichier trouvé dans le Drive :", existant[0])
else:
    envoye = files.upload()
    nom = list(envoye.keys())[0]
    extension = ".xlsx" if nom.lower().endswith((".xlsx", ".xls")) else ".csv"
    with open(os.path.join(DOSSIER, "formations" + extension), "wb") as f:
        f.write(envoye[nom])
    formations = lire(nom)
    print("Copie enregistrée dans le Drive : formations" + extension)

print(len(formations), "formations chargées")
formations[["ecole", "nom", "frais_national", "frais_international", "source_url"]]

Fichier trouvé dans le Drive : formations.xlsx
6 formations chargées


,ecole,nom,frais_national,frais_international,source_url
0,IIT,Génie Civil,556DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-civil/
1,IIT,Génie Mécanique,611DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-mecani...
2,IIT,Génie Informatique,778DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-inform...
3,IIT,Génie Industriel,611DT/9 mois*,234€/9 mois*,https://iit.tn/filieres-ingenieur/genie-indust...
4,IIT,Génie des procédés,611DT/9 mois*,234€/9 mois*,https://iit.tn/genie-des-procedes/
5,ISB,License in Business Administration (Licence en...,,,https://isb.tn/program-lba/


## 5. Les deux consignes données au modèle (identique)

In [6]:
CHAMPS = [
    ("type", "Type"), ("duree_annees", "Durée (années)"), ("conditions_acces", "Conditions d'accès"),
    ("debouches", "Débouchés"), ("description", "Description"),
    ("frais_national", "Frais (étudiants tunisiens)"), ("frais_international", "Frais (étudiants internationaux)"),
    ("source_url", "Source"),
]

def bloc_formation(ligne):
    texte = f"### {ligne['ecole']} — {ligne['nom']}\n"
    for col, label in CHAMPS:
        valeur = str(ligne.get(col, "")).strip()
        texte += f"- {label} : {valeur if valeur else 'non communiqué'}\n"
    return texte

CONTEXTE = "\n".join(bloc_formation(l) for _, l in formations.iterrows())

REGLES_LANGUE = ("Réponds dans la langue et l'écriture de l'étudiant : français, tounsi en lettres arabes, "
                 "ou Arabizi (tounsi en lettres latines avec 3, 7, 9, 5, 8, 2). Sois clair, court et bienveillant.")

SYSTEM_CONTEXTE = (
    "Tu es SmartCampus AI, l'assistant d'orientation de l'IIT et de l'ISB (Sfax, Tunisie).\n"
    "1. Réponds UNIQUEMENT à partir des informations officielles ci-dessous.\n"
    "2. Si une information est « non communiqué » ou absente, dis-le clairement et propose de contacter "
    "l'établissement. N'invente jamais un chiffre.\n"
    "3. " + REGLES_LANGUE + "\n\n=== INFORMATIONS OFFICIELLES ===\n" + CONTEXTE
)

SYSTEM_SANS_CONTEXTE = "Tu es SmartCampus AI, un assistant pour les étudiants tunisiens. " + REGLES_LANGUE

SYSTEMES = {"contexte": SYSTEM_CONTEXTE, "sans_contexte": SYSTEM_SANS_CONTEXTE}
print("Contexte : environ", len(SYSTEM_CONTEXTE) // 4, "tokens")

Contexte : environ 2266 tokens


## 6. Les questions de test (identiques)
- **mode `contexte`** (C) : questions sur les formations, avec les données officielles → **SA-33**.
- **mode `sans_contexte`** (D) : compréhension du **dialecte** seul → **SA-34**.

In [7]:
Q_CONTEXTE = [
    # (langue, question, mots_attendus, piege)   — 28 questions, 5 langues, 1 piège par langue
    # --- Français (5) ---
    ("Français", "Quels sont les frais du Génie Informatique à l'IIT ?", "778", "non"),
    ("Français", "Quelles sont les conditions d'accès au Génie Mécanique ?", "préparatoire|prépa|MP|licence", "non"),
    ("Français", "Combien d'années dure la licence Business Administration de l'ISB ?", "3", "non"),
    ("Français", "Quels sont les frais d'inscription en plus des frais de scolarité à l'IIT ?", "800", "non"),
    ("Français", "Quels sont les frais de la licence Business Administration à l'ISB ?", "", "oui"),
    # --- Anglais (5) ---
    ("Anglais",  "What career opportunities are there after Industrial Engineering at IIT?", "logistic|quality|supply|production|consult", "non"),
    ("Anglais",  "Is the Business Administration bachelor at ISB taught in English?", "yes|english", "non"),
    ("Anglais",  "Which specializations are offered in Computer Engineering at IIT?", "GLID|ARSI|SDIA", "non"),
    ("Anglais",  "Which accreditation does the IIT Computer Engineering program have?", "ASIIN|Euro-Inf", "non"),
    ("Anglais",  "Does IIT offer scholarships for engineering students?", "", "oui"),
    # --- Tounsi en lettres arabes (5) ---
    ("Tounsi",   "قداش معاليم الهندسة المدنية في IIT ؟", "556|600", "non"),
    ("Tounsi",   "شنوة الخدمة اللي نجم نخدمها بعد الهندسة الصناعية ؟", "logisti|qualit|supply|لوجست|جودة|إنتاج", "non"),
    ("Tounsi",   "شنية الشروط باش نقرا هندسة الإعلامية ؟", "préparatoire|prépa|licence|إعداد|إجازة|ليسانس", "non"),
    ("Tounsi",   "قداش تزيد المعاليم بعد العام الأول ؟", "5 %|5%|10%", "non"),
    ("Tounsi",   "الهندسة الميكانيكية تتقرا بالفرنساوي ولا بالأنقليزي ؟", "", "oui"),
    # --- Arabizi (9) ---
    ("Arabizi",  "9adeh soum genie info fel IIT ?", "778", "non"),
    ("Arabizi",  "chnouma les parcours elli fama fi genie informatique ?", "GLID|ARSI|SDIA", "non"),
    ("Arabizi",  "3aslema, billehi n7eb nes2lek 3al genie info, chnouma el conditions bech nod5ol ?", "préparatoire|prépa|licence", "non"),
    ("Arabizi",  "salam, ena 3andi bac math w n7eb na9ra fel IIT, chnouma el 5ayarat elli 3andi ?", "prépa|préparatoire|licence", "non"),

    ("Arabizi",  "n7eb n5dem fel cybersécurité, ama parcours ynasbni fi genie info ?", "ARSI", "non"),
    ("Arabizi",  "famma frais d'inscription zeyda walla le ?", "800", "non"),
    ("Arabizi",  "9adeh soum genie electrique fel IIT ?", "", "oui"),
    # --- Mélange de langues (4) ---
    ("Mélange",  "Bonjour, nheb n3ref les débouchés mta3 génie civil", "structure|construction|infrastructure|bureau", "non"),
    ("Mélange",  "Hi, n7eb na9ra business in English, famma programme fel ISB ?", "Business Administration|English|anglais", "non"),
    ("Mélange",  "Salut, genie des procédés yod5el fih el pétrole walla le ?", "pétrol|petrol|énergie|energie", "non"),
    ("Mélange",  "Hello, 9addech men 3am yodhher genie civil ?", "", "oui"),
]

Q_SANS_CONTEXTE = [
    ("Arabizi",  "chnowa ma3neha 'barcha' bel fransa ?", "beaucoup", "non"),
    ("Arabizi",  "traduis en français : ena mech fehem chay", "comprends", "non"),
    ("Tounsi",   "شنوة معناها 'يعيشك' بالفرنسية ؟", "merci", "non"),
    ("Arabizi",  "kifech nqoul 'je suis fatigué' bel tounsi ?", "ta3ban|t3ban|تعبان|3yit|عييت", "non"),
    ("Arabizi",  "chnowa ma3neha 'taw' bel fransa ?", "maintenant|tout de suite|now", "non"),
    ("Tounsi",   "ترجم للفرنسية : نحب نقرا في الجامعة", "étudier|université|universite", "non"),
    ("Arabizi",  "9oulli bel arabizi : comment tu t'appelles ?", "esmek|ismek|chesmek|chnowa esmek|ism", "non"),
    ("Arabizi",  "3lech fel arabizi nekteb '3' w '7' ?", "ع|ح|lettre|letter|7arf|حرف", "non"),
    ("Mélange",  "rani stressé barcha 3al bac, 3andek nasi7a ?", "révis|organis|repos|sommeil|respir|plan|نوم", "non"),
    ("Arabizi",  "jawebni bel tounsi : chnowa a7sen wa9t bech na9ra fel nhar ?", "sbe7|sba7|matin|lil|صباح|ليل", "non"),
]

lignes = [(f"C{i}", "contexte", *q) for i, q in enumerate(Q_CONTEXTE, 1)] + \
         [(f"D{i}", "sans_contexte", *q) for i, q in enumerate(Q_SANS_CONTEXTE, 1)]
df_questions = pd.DataFrame(lignes, columns=["id", "mode", "langue", "question", "mots_attendus", "piege"])
print(df_questions["mode"].value_counts())
df_questions.head()

mode
contexte         26
sans_contexte    10
Name: count, dtype: int64


,id,mode,langue,question,mots_attendus,piege
0,C1,contexte,Français,Quels sont les frais du Génie Informatique à l...,778,non
1,C2,contexte,Français,Quelles sont les conditions d'accès au Génie M...,préparatoire|prépa|MP|licence,non
2,C3,contexte,Français,Combien d'années dure la licence Business Admi...,3,non
3,C4,contexte,Français,Quels sont les frais d'inscription en plus des...,800,non
4,C5,contexte,Français,Quels sont les frais de la licence Business Ad...,,oui


## 7. Le corpus de perplexité (identique au notebook de perplexité)

In [8]:
CORPUS = os.path.join(DOSSIER, "corpus_perplexite.csv")
if not os.path.exists(CORPUS):
    envoye = files.upload()                      # choisissez corpus_perplexite.csv
    nom = list(envoye.keys())[0]
    with open(CORPUS, "wb") as f:
        f.write(envoye[nom])
    print("Copie enregistrée dans le Drive :", CORPUS)
else:
    print("Corpus trouvé dans le Drive :", CORPUS, "(pour le changer, remplacez ce fichier dans le Drive)")

corpus = pd.read_csv(CORPUS, dtype=str).fillna("")
corpus = corpus[corpus["phrase"].str.strip() != ""].reset_index(drop=True)
corpus["phrase_id"] = corpus.index.astype(str)
print(corpus["langue"].value_counts())

# Vérification sécurisée de la colonne 'source'
if "source" in corpus.columns:
    if (corpus["source"].str.contains("exemple", case=False)).any():
        print("⚠️ Le corpus contient encore des phrases d'exemple : remplacez-les par de vraies phrases avant de présenter les résultats.")
else:
    print("Information : Pas de colonne 'source' détectée dans le corpus.")

Corpus trouvé dans le Drive : /content/drive/MyDrive/SmartCampus_LLM/corpus_perplexite.csv (pour le changer, remplacez ce fichier dans le Drive)
langue
Tounsi      50
Arabizi     50
Français    50
Anglais     50
Name: count, dtype: int64
Information : Pas de colonne 'source' détectée dans le corpus.


## 8. Les 2 modèles à tester
Mêmes réglages de génération que pour les 3 anciens modèles (300 tokens maximum, température 0,3).

In [9]:
MODELES = [
    "CohereLabs/aya-expanse-8b",  # modèle de base Cohere, multilingue (accès restreint : accepter les conditions)
    "google/gemma-4-12B-it",      # Gemma 4 12B, multimodal, Apache 2.0
]

MAX_NEW_TOKENS = 300
TEMPERATURE = 0.3

## 9. Fonctions (chargement, génération, score, perplexité)

Différences avec les anciens notebooks, nécessaires pour ces 2 modèles :
- **Gemma 4 est multimodal** : il se charge avec `AutoModelForMultimodalLM` (essayé automatiquement).
- **Gemma et Aya calculent en bfloat16** (le format de leur entraînement) : en float16 leurs calculs peuvent déborder (résultats `NaN`).
- Si un jour on teste un **adaptateur LoRA** sans tokenizer, celui de son modèle de base est pris automatiquement (prévu pour le sprint 2).
- **Jeton de début (BOS)** : Gemma et Aya en ont un, et ils fonctionnent mal sans. On le met devant le texte.
  Les anciens modèles Qwen n'en ont pas : pour eux, le calcul est **identique** à l'ancien, donc les chiffres restent comparables.
- La « réflexion » de Gemma est désactivée (`enable_thinking=False`) et retirée de la réponse si elle apparaît.

In [10]:
import torch, transformers
from packaging.version import Version
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

CLE_DTYPE = "dtype" if Version(transformers.__version__) >= Version("4.56") else "torch_dtype"

def est_gemma(nom):
    return "gemma" in nom.lower()

def config_4bits(nom):
    besoin_bf16 = est_gemma(nom) or "aya" in nom.lower()
    calcul = torch.bfloat16 if (NATIF_BF16 or besoin_bf16) else torch.float16
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=calcul)
    return bnb, calcul

def modele_de_base(nom):
    """Pour un adaptateur LoRA : le nom du modèle de base écrit dans adapter_config.json (sinon None)."""
    try:
        from peft import PeftConfig
        return PeftConfig.from_pretrained(nom).base_model_name_or_path
    except Exception:
        return None

def charger_tokenizer(nom):
    tok = None
    for source in (nom, modele_de_base(nom)):
        if not source:
            continue
        try:
            tok = AutoTokenizer.from_pretrained(source)
            break
        except Exception as e:
            print(f"  tokenizer non trouvé dans {source} : {e}")
    if tok is None or tok.chat_template is None:          # Gemma 4 : le modèle de conversation est dans le processor
        from transformers import AutoProcessor
        proc = AutoProcessor.from_pretrained(nom)
        if tok is None:
            tok = getattr(proc, "tokenizer", proc)
        if getattr(tok, "chat_template", None) is None:
            tok.chat_template = proc.chat_template
    return tok

def charger(nom):
    tok = charger_tokenizer(nom)
    bnb, calcul = config_4bits(nom)
    classes = []
    if est_gemma(nom):
        classes += [getattr(transformers, c) for c in ("AutoModelForMultimodalLM", "AutoModelForImageTextToText")
                    if hasattr(transformers, c)]
    classes.append(AutoModelForCausalLM)
    erreurs = []
    for classe in classes:
        try:
            mod = classe.from_pretrained(nom, quantization_config=bnb, device_map="auto", **{CLE_DTYPE: calcul})
            mod.eval()
            print(f"  chargé avec {classe.__name__}, calcul en {calcul}, "
                  f"mémoire GPU utilisée : {torch.cuda.memory_allocated() / 1e9:.1f} Go")
            return tok, mod
        except Exception as e:
            erreurs.append(f"{classe.__name__} : {e}")
    raise RuntimeError("\n".join(erreurs))

def construire_prompt(tok, question, systeme):
    messages = [{"role": "system", "content": systeme}, {"role": "user", "content": question}]
    for extra in (dict(enable_thinking=False), dict()):   # enable_thinking=False : pas de « réflexion »
        try:
            return tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **extra)
        except Exception:
            pass
    fusion = [{"role": "user", "content": systeme + "\n\nQuestion de l'étudiant : " + question}]
    return tok.apply_chat_template(fusion, tokenize=False, add_generation_prompt=True)

def nettoyer(texte):
    texte = re.sub(r"<think>.*?</think>", "", texte, flags=re.S)              # Qwen3
    texte = re.sub(r"<\|channel>.*?<channel\|>", "", texte, flags=re.S)      # Gemma 4 (si la réflexion apparaît)
    return texte.strip()

@torch.no_grad()
def generer(tok, mod, question, systeme):
    # add_special_tokens=False : le modèle de conversation contient déjà le jeton de début (pas de doublon)
    entrees = tok(construire_prompt(tok, question, systeme), return_tensors="pt", add_special_tokens=False).to(mod.device)
    debut = time.time()
    sorties = mod.generate(**entrees, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                           top_p=0.9, pad_token_id=tok.pad_token_id or tok.eos_token_id)
    texte = tok.decode(sorties[0][entrees["input_ids"].shape[1]:], skip_special_tokens=True)
    return nettoyer(texte), time.time() - debut

def score_auto(reponse, mots_attendus, piege):
    rep = reponse.lower()
    if piege == "oui":   # 1 si le modèle n'invente pas de prix ou de durée
        invente = bool(re.search(r"\d{2,}\s*(dt|dinar|€|eur|دينار)|\d+\s*(ans\b|an\b|années|سنوات|سنين)", rep))
        return 0 if invente else 1
    mots = [m.strip().lower() for m in str(mots_attendus).split("|") if m.strip()]
    return 1 if any(m in rep for m in mots) else 0

@torch.no_grad()
def nll_phrase(tok, mod, phrase):
    """NLL totale (en nats) des tokens de la phrase, et leur nombre."""
    bos = [tok.bos_token_id] if tok.bos_token_id is not None else []      # Qwen : aucun → même calcul qu'avant
    prefixe = bos + tok("\n", add_special_tokens=False)["input_ids"]
    ids = tok(phrase, add_special_tokens=False)["input_ids"]
    x = torch.tensor([prefixe + ids], device=mod.device)
    logits = mod(input_ids=x).logits[0, :-1].float()                       # prédiction du token suivant
    pertes = torch.nn.functional.cross_entropy(logits, x[0, 1:], reduction="none")
    nll = pertes[len(prefixe) - 1:].sum().item()                           # on n'évalue que la phrase
    return nll, len(ids)

## 10. Lancer les 2 tests (sauvegarde régulière dans le Drive)
Pour chaque modèle : chargement **une fois**, puis **A.** les 36 questions, puis **B.** les phrases du corpus.
Si la session se coupe, relancez les sections 1b à 10 : le travail reprend là où il s'était arrêté.

In [11]:
torch.manual_seed(0)

for nom in MODELES:
    deja = {(r["modele"], str(r["id"])) for r in resultats}
    q_a_faire = df_questions[[(nom, i) not in deja for i in df_questions["id"]]]
    faits = {str(r["phrase_id"]) for r in details if r["modele"] == nom}
    p_a_faire = corpus[~corpus["phrase_id"].isin(faits)]
    if q_a_faire.empty and p_a_faire.empty:
        print("✔ déjà terminé :", nom)
        continue

    print("\n" + "=" * 70 + f"\n{nom} : {len(q_a_faire)} questions + {len(p_a_faire)} phrases")
    try:
        tok, mod = charger(nom)
    except Exception as e:
        print("❌ Impossible de charger ce modèle :\n", e)
        print("   → Aya : avez-vous accepté les conditions d'aya-expanse-8b et collé votre token ?")
        continue

    # ---- A. Comparaison : réponses aux questions (SA-33 / SA-34) ----
    for _, q in q_a_faire.iterrows():
        try:
            reponse, duree = generer(tok, mod, q["question"], SYSTEMES[q["mode"]])
        except Exception as e:
            reponse, duree = f"ERREUR : {e}", 0.0
        s = score_auto(reponse, q["mots_attendus"], q["piege"])
        resultats.append({"modele": nom, "id": q["id"], "mode": q["mode"], "langue": q["langue"],
                          "question": q["question"], "piege": q["piege"], "reponse": reponse,
                          "temps_s": round(duree, 1), "score_auto": s})
        sauvegarder()
        print(f"[{q['id']}] ({q['langue']}) {q['question']}\n→ {reponse[:250]}\n   score={s}  temps={duree:.1f}s\n")

    # ---- B. Perplexité : phrases du corpus ----
    for k, (_, r) in enumerate(p_a_faire.iterrows(), 1):
        nll, n_tok = nll_phrase(tok, mod, r["phrase"])
        if math.isnan(nll) or math.isinf(nll):
            print(f"⚠️ NLL invalide pour la phrase {r['phrase_id']} : vérifiez le type de calcul (bfloat16).")
        details.append({"modele": nom, "phrase_id": r["phrase_id"], "langue": r["langue"], "nll": nll,
                        "tokens": n_tok, "caracteres": len(r["phrase"]), "mots": len(r["phrase"].split())})
        if k % 20 == 0:
            sauvegarder_ppl()
            print(f"  perplexité : {k}/{len(p_a_faire)} phrases")
    sauvegarder_ppl()

    # Libérer vraiment la mémoire GPU et le disque avant le modèle suivant
    del mod, tok
    gc.collect()
    torch.cuda.empty_cache()
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub"), ignore_errors=True)

print("✅ Terminé.")


CohereLabs/aya-expanse-8b : 36 questions + 200 phrases
  tokenizer non trouvé dans CohereLabs/aya-expanse-8b : We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
❌ Impossible de charger ce modèle :
 We couldn't connect to 'https://huggingface.co' to load the files, and couldn't find them in the cached files.
Check your internet connection or see how to run the library in offline mode at 'https://huggingface.co/docs/transformers/installation#offline-mode'.
   → Aya : avez-vous accepté les conditions d'aya-expanse-8b et collé votre token ?

google/gemma-4-12B-it : 36 questions + 200 phrases


config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

  chargé avec AutoModelForMultimodalLM, calcul en torch.bfloat16, mémoire GPU utilisée : 8.2 Go
[C1] (Français) Quels sont les frais du Génie Informatique à l'IIT ?
→ Les frais pour le cursus Génie Informatique à l'IIT sont les suivants :

*   **Étudiants tunisiens :** 778 DT / 9 mois*
*   **Étudiants internationaux :** 234 € / 9 mois*

Ces frais sont indicatifs et valables pour la 1re année seulement (une augment
   score=1  temps=50.9s

[C2] (Français) Quelles sont les conditions d'accès au Génie Mécanique ?
→ Pour accéder au cursus de Génie Mécanique à l'IIT, les conditions sont les suivantes :

*   **Diplôme de préparatoire** (MP/PC/PT) ;
*   **OU une licence** dans l'une des spécialités suivantes : maintenance industrielle, électromécanique, conception 
   score=1  temps=57.2s

[C3] (Français) Combien d'années dure la licence Business Administration de l'ISB ?
→ La licence en Administration des Affaires (Business Administration) de l'ISB dure 3 ans.
   score=1  temps=33.5s

[C4] (

## 11. Résultats de la comparaison (5 modèles)
Fonctionne aussi **sans GPU** : exécutez seulement les sections **3**, **11**, **12**, **13** et **14**.

In [12]:
df = pd.read_csv(FICHIER_RESULTATS, dtype={"id": str})
df["score_auto"] = pd.to_numeric(df["score_auto"], errors="coerce")

resume = df.pivot_table(index="modele", columns="mode", values="score_auto", aggfunc="mean")
resume = (resume * 100).round(0)
resume["temps_moyen_s"] = df.groupby("modele")["temps_s"].mean().round(1)
pieges = df[df["piege"] == "oui"].groupby("modele")["score_auto"].mean()
resume["pieges_evites_%"] = (pieges * 100).round(0)

par_langue = (df.pivot_table(index="modele", columns="langue", values="score_auto", aggfunc="mean") * 100).round(0)

print("Scores en % (contexte = SA-33, sans_contexte = SA-34)")
display(resume)
print("Scores en % par langue")
display(par_langue)

Scores en % (contexte = SA-33, sans_contexte = SA-34)


mode,contexte,sans_contexte,temps_moyen_s,pieges_evites_%
modele,,,,
Artess99/derja-qwen3-8b-derja-v2,88.0,10.0,13.9,100.0
Qwen/Qwen2.5-7B-Instruct,88.0,10.0,11.8,100.0
Qwen/Qwen3-8B,77.0,20.0,20.8,100.0
google/gemma-4-12B-it,88.0,70.0,48.8,60.0


Scores en % par langue


langue,Anglais,Arabizi,Français,Mélange,Tounsi
modele,,,,,
Artess99/derja-qwen3-8b-derja-v2,100.0,29.0,100.0,80.0,86.0
Qwen/Qwen2.5-7B-Instruct,80.0,36.0,100.0,80.0,86.0
Qwen/Qwen3-8B,100.0,43.0,100.0,60.0,43.0
google/gemma-4-12B-it,100.0,71.0,100.0,80.0,86.0


## 12. Résultats de la perplexité (5 modèles)
- **BPC** (bits par caractère) : mesure principale, **comparable entre modèles** même si leurs tokenizers sont différents. Plus bas = mieux.
- La perplexité par mot est donnée pour information : elle dépend beaucoup du tokenizer.

In [13]:
dfp = pd.read_csv(FICHIER_PPL)
g = dfp.groupby(["modele", "langue"])[["nll", "tokens", "caracteres", "mots"]].sum()
g["BPC"] = g["nll"] / math.log(2) / g["caracteres"]
g["perplexite_par_mot"] = (g["nll"] / g["mots"]).apply(math.exp)
g["perplexite_par_token"] = (g["nll"] / g["tokens"]).apply(math.exp)
g["phrases"] = dfp.groupby(["modele", "langue"]).size()
res_ppl = g.reset_index()

bpc = res_ppl.pivot(index="modele", columns="langue", values="BPC").round(3)
ppl_mot = res_ppl.pivot(index="modele", columns="langue", values="perplexite_par_mot").round(1)
print("Bits par caractère (plus bas = mieux)"); display(bpc)
print("Perplexité par mot (plus bas = mieux)"); display(ppl_mot)

Bits par caractère (plus bas = mieux)


langue,Anglais,Arabizi,Français,Tounsi
modele,,,,
Artess99/derja-qwen3-8b-derja-v2,1.557,4.161,1.445,3.404
Qwen/Qwen2.5-7B-Instruct,1.372,4.353,1.275,3.352
Qwen/Qwen3-8B,1.393,4.339,1.367,3.558
google/gemma-4-12B-it,2.556,6.837,3.049,5.994


Perplexité par mot (plus bas = mieux)


langue,Anglais,Arabizi,Français,Tounsi
modele,,,,
Artess99/derja-qwen3-8b-derja-v2,344.9,1.822829e+07,402.5,2.125401e+05
Qwen/Qwen2.5-7B-Instruct,172.2,3.946502e+07,198.7,1.763430e+05
Qwen/Qwen3-8B,186.4,3.737424e+07,290.7,3.705713e+05
google/gemma-4-12B-it,14693.0,8.542432e+11,314408.0,2.403770e+09


## 13. Synthèse et classement
Un tableau unique par modèle. Le **rang moyen** combine 6 critères à poids égal :
réponses avec données (SA-33), dialecte seul (SA-34), score Arabizi, score Tounsi, BPC Arabizi, BPC Tounsi.
Rang 1 = meilleur. Le temps est affiché mais **n'entre pas** dans le classement.

⚠️ Le score automatique vérifie seulement la présence de mots attendus sur 36 questions : **relisez les réponses**
(feuille `reponses` de l'Excel, colonnes de notes 1 à 5) avant de conclure.

In [14]:
synthese = pd.DataFrame({
    "SA-33 contexte %":      resume.get("contexte"),
    "SA-34 dialecte seul %": resume.get("sans_contexte"),
    "Arabizi %":             par_langue.get("Arabizi"),
    "Tounsi %":              par_langue.get("Tounsi"),
    "Mélange %":             par_langue.get("Mélange"),
    "BPC Arabizi":           bpc.get("Arabizi"),
    "BPC Tounsi":            bpc.get("Tounsi"),
    "BPC Français":          bpc.get("Français"),
    "pièges évités %":       resume.get("pieges_evites_%"),
    "temps moyen (s)":       resume.get("temps_moyen_s"),
})

plus_haut_mieux = ["SA-33 contexte %", "SA-34 dialecte seul %", "Arabizi %", "Tounsi %"]
plus_bas_mieux = ["BPC Arabizi", "BPC Tounsi"]
rangs = pd.concat([synthese[plus_haut_mieux].rank(ascending=False, method="min"),
                   synthese[plus_bas_mieux].rank(ascending=True, method="min")], axis=1)
synthese["rang moyen"] = rangs.mean(axis=1).round(2)
synthese["nouveau"] = ["✅" if m in MODELES else "" for m in synthese.index]
synthese = synthese.sort_values("rang moyen")
display(synthese)

meilleur = synthese.index[0]
print(f"🏆 Meilleur rang moyen : {meilleur}")
for critere in plus_haut_mieux:
    print(f"   meilleur en {critere:24s}: {synthese[critere].idxmax()}")
for critere in plus_bas_mieux:
    print(f"   meilleur en {critere:24s}: {synthese[critere].idxmin()}")

,SA-33 contexte %,SA-34 dialecte seul %,Arabizi %,Tounsi %,Mélange %,BPC Arabizi,BPC Tounsi,BPC Français,pièges évités %,temps moyen (s),rang moyen,nouveau
modele,,,,,,,,,,,,
Artess99/derja-qwen3-8b-derja-v2,88.0,10.0,29.0,86.0,80.0,4.161,3.404,1.445,100.0,13.9,2.00,
Qwen/Qwen2.5-7B-Instruct,88.0,10.0,36.0,86.0,80.0,4.353,3.352,1.275,100.0,11.8,2.00,
google/gemma-4-12B-it,88.0,70.0,71.0,86.0,80.0,6.837,5.994,3.049,60.0,48.8,2.00,✅
Qwen/Qwen3-8B,77.0,20.0,43.0,43.0,60.0,4.339,3.558,1.367,100.0,20.8,2.83,


🏆 Meilleur rang moyen : Artess99/derja-qwen3-8b-derja-v2
   meilleur en SA-33 contexte %        : Artess99/derja-qwen3-8b-derja-v2
   meilleur en SA-34 dialecte seul %   : google/gemma-4-12B-it
   meilleur en Arabizi %               : google/gemma-4-12B-it
   meilleur en Tounsi %                : Artess99/derja-qwen3-8b-derja-v2
   meilleur en BPC Arabizi             : Artess99/derja-qwen3-8b-derja-v2
   meilleur en BPC Tounsi              : Qwen/Qwen2.5-7B-Instruct


## 14. Export Excel

In [15]:
notation = df.copy()
for col in ["note_comprehension_1a5", "note_bonne_langue_1a5", "note_fidelite_donnees_1a5", "note_clarte_1a5", "commentaire"]:
    notation[col] = ""

FICHIER_XLSX = os.path.join(DOSSIER, "comparaison_perplexite_5_modeles.xlsx")
with pd.ExcelWriter(FICHIER_XLSX, engine="openpyxl") as w:
    synthese.to_excel(w, sheet_name="synthese")
    resume.to_excel(w, sheet_name="resume_reponses")
    par_langue.to_excel(w, sheet_name="reponses_par_langue")
    bpc.to_excel(w, sheet_name="BPC")
    ppl_mot.to_excel(w, sheet_name="perplexite_par_mot")
    notation.to_excel(w, sheet_name="reponses", index=False)
    df_questions.to_excel(w, sheet_name="questions", index=False)

files.download(FICHIER_XLSX)
print("✅ Excel enregistré dans le Drive et téléchargé :", FICHIER_XLSX)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Excel enregistré dans le Drive et téléchargé : /content/drive/MyDrive/SmartCampus_LLM/comparaison_perplexite_5_modeles.xlsx


In [16]:
# Test : BPC du modèle de BASE Gemma-4-12B (sans "-it"), en anglais et en français
# But : vérifier si le BPC élevé de gemma-4-12B-it vient du format chat (et non d'une mauvaise connaissance de la langue)
import gc, math

BASE = "google/gemma-4-12B"
LANGUES = ["Anglais", "Français"]            # 2 langues suffisent pour la vérification (100 phrases)

tok_b, mod_b = charger(BASE)

# 1) Contrôle rapide sur une phrase très simple : un bon modèle doit donner un BPC bas (environ 0,5 à 1,5)
test = "The weather is nice today and I want to go for a walk."
nll, n = nll_phrase(tok_b, mod_b, test)
print(f"Phrase de contrôle : BPC = {nll / math.log(2) / len(test):.3f}")

# 2) Les phrases du corpus
lignes = []
for _, r in corpus[corpus["langue"].isin(LANGUES)].iterrows():
    nll, n = nll_phrase(tok_b, mod_b, r["phrase"])
    lignes.append({"langue": r["langue"], "nll": nll, "caracteres": len(r["phrase"])})

res = pd.DataFrame(lignes).groupby("langue")[["nll", "caracteres"]].sum()
res["BPC base (sans -it)"] = (res["nll"] / math.log(2) / res["caracteres"]).round(3)
res["BPC -it (déjà mesuré)"] = pd.Series({"Anglais": 2.556, "Français": 3.049})
res["Qwen2.5-7B (pour repère)"] = pd.Series({"Anglais": 1.372, "Français": 1.275})
print(res[["BPC base (sans -it)", "BPC -it (déjà mesuré)", "Qwen2.5-7B (pour repère)"]])

del mod_b; gc.collect(); torch.cuda.empty_cache()    # libère la mémoire GPU

config.json:   0%|          | 0.00/4.38k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

  chargé avec AutoModelForMultimodalLM, calcul en torch.bfloat16, mémoire GPU utilisée : 8.2 Go
Phrase de contrôle : BPC = 0.851
          BPC base (sans -it)  BPC -it (déjà mesuré)  Qwen2.5-7B (pour repère)
langue                                                                        
Anglais                 1.234                  2.556                     1.372
Français                1.325                  3.049                     1.275


In [17]:
# =====================================================================
# Test : BPC du modèle de BASE Gemma-4-12B (sans "-it") en TOUNSI et ARABIZI
# + comparaison avec les autres modèles + sauvegarde Drive / téléchargement
# =====================================================================
import gc, math, os, glob
from datetime import datetime
from google.colab import files

BASE = "google/gemma-4-12B"
LANGUES = ["Tounsi", "Arabizi"]

# BPC déjà mesurés (notebooks précédents) pour comparer
REPERES = {
    "Gemma-4-12B-it":      {"Tounsi": 5.994, "Arabizi": 6.837},
    "Qwen2.5-7B-Instruct": {"Tounsi": 3.352, "Arabizi": 4.353},
    "Qwen3-8B":            {"Tounsi": 3.558, "Arabizi": 4.339},
    "Artess99 (LoRA)":     {"Tounsi": 3.404, "Arabizi": 4.161},
}

# 1) Chargement (si le modèle n'est plus en mémoire)
if "mod_b" not in globals():
    tok_b, mod_b = charger(BASE)

# 2) Calcul du BPC sur les phrases du corpus
lignes = []
a_faire = corpus[corpus["langue"].isin(LANGUES)]
for i, (_, r) in enumerate(a_faire.iterrows(), 1):
    nll, n = nll_phrase(tok_b, mod_b, r["phrase"])
    lignes.append({"langue": r["langue"], "nll": nll, "caracteres": len(r["phrase"])})
    if i % 20 == 0:
        print(f"  {i}/{len(a_faire)} phrases")

res_dialecte = pd.DataFrame(lignes).groupby("langue")[["nll", "caracteres"]].sum()
res_dialecte["Gemma-4-12B base (sans -it)"] = (res_dialecte["nll"] / math.log(2) / res_dialecte["caracteres"]).round(3)
for nom, vals in REPERES.items():
    res_dialecte[nom] = pd.Series(vals)
tableau_d = res_dialecte.drop(columns=["nll", "caracteres"])
print("\nBPC (plus bas = mieux)\n", tableau_d)

meilleur = tableau_d.idxmin(axis=1)
for langue, nom in meilleur.items():
    print(f"🏆 meilleur en {langue} : {nom}")

# 3) Sauvegarde dans le Drive
tableau_d["date"] = datetime.now().strftime("%Y-%m-%d %H:%M")
csv_path  = os.path.join(DOSSIER, "bpc_gemma_base_dialecte.csv")
xlsx_path = os.path.join(DOSSIER, "bpc_gemma_base_dialecte.xlsx")
tableau_d.to_csv(csv_path, encoding="utf-8")
tableau_d.to_excel(xlsx_path, sheet_name="BPC_gemma_base_dialecte")
print("\n✅ Enregistré dans le Drive :\n  ", csv_path, "\n  ", xlsx_path)

#    + feuille ajoutée au grand fichier Excel de comparaison (s'il est dans le dossier)
for chemin in glob.glob(os.path.join(DOSSIER, "comparaison*.xlsx")):
    with pd.ExcelWriter(chemin, engine="openpyxl", mode="a", if_sheet_exists="replace") as w:
        tableau_d.to_excel(w, sheet_name="BPC_gemma_base_dialecte")
    print("✅ Feuille 'BPC_gemma_base_dialecte' ajoutée dans :", chemin)

# 4) Copie sur votre ordinateur
files.download(xlsx_path)

# 5) Libère la mémoire GPU
del mod_b; gc.collect(); torch.cuda.empty_cache()

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

  chargé avec AutoModelForMultimodalLM, calcul en torch.bfloat16, mémoire GPU utilisée : 8.2 Go
  20/100 phrases
  40/100 phrases
  60/100 phrases
  80/100 phrases
  100/100 phrases

BPC (plus bas = mieux)
          Gemma-4-12B base (sans -it)  Gemma-4-12B-it  Qwen2.5-7B-Instruct  \
langue                                                                      
Arabizi                        3.628           6.837                4.353   
Tounsi                         2.723           5.994                3.352   

         Qwen3-8B  Artess99 (LoRA)  
langue                              
Arabizi     4.339            4.161  
Tounsi      3.558            3.404  
🏆 meilleur en Arabizi : Gemma-4-12B base (sans -it)
🏆 meilleur en Tounsi : Gemma-4-12B base (sans -it)

✅ Enregistré dans le Drive :
   /content/drive/MyDrive/SmartCampus_LLM/bpc_gemma_base_dialecte.csv 
   /content/drive/MyDrive/SmartCampus_LLM/bpc_gemma_base_dialecte.xlsx
✅ Feuille 'BPC_gemma_base_dialecte' ajoutée dans : /content/d

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>